# Lecture 15: Finding the structure automatically

**The challenge.** Every exact solve since Lecture 12 needed something more
than the problem: a smoothing parameter and a schedule to lower it, a
penalty weight and a schedule to raise it, and above all the *structure* of
the answer. Which arcs occur, in what order, held at which bounds? For
Goddard's rocket that is one line. For a six-joint robot with velocity,
acceleration and torque limits it would be dozens of junctions, and writing
them down requires knowing the answer. The goal of this lecture is the
interface the earlier lectures have been using as a black box: the problem,
its parameters and a distribution to draw initial costates from, and
nothing else.

**After this lecture you can**

- explain each stage of `Homotopy.solve`: regularize, sample, continue in
  phases, read the structure, jump, accept;
- read a smoothed solution the way the reader does, and say why the exact
  problem can be solved long before the smoothing is small;
- interpret a `HomotopyResult`, its continuation log and its failure modes.

## 1. From hand-made structures to automatic ones

In the shooting codes of the 1970s and 80s, such as Oberle's BNDSCO from the
Munich school (Lecture 9), the user supplied the switching structure, and
getting it right was the art. A popular remedy from the 1990s, described by
Oskar von Stryk and Roland Bulirsch in 1992, was hybrid: solve a coarse
*direct* (discretized) version first, read the structure and the costates
from it, then refine with the indirect method. This package takes a
different route and never seeds from a direct method. Everything is read
from *indirect* solutions of nearby, easier problems: the smoothed and
penalized problems of Lectures 10 and 13, which have no switching structure
to get wrong. Their solutions show where the structure is. Lectures 10 and 14
already did each step by hand; here they run together.

## 2. The stages

1. **Regularize.** `regularize(problem)` adds
   $\varepsilon((u - m)/h)^2/2$ for each bang-bang control, and for each path
   constraint an exterior penalty weight $\rho_c$ and an offset $o_c$ (the
   limit becomes $g_c - o_c \le 0$), all runtime parameters (`eps_reg`,
   `rho_reg_<name>`, `off_reg_<name>`). `exact(problem)` is the problem with
   any penalty weights removed.
2. **Sample.** Single shooting on the regularized problem at
   $\varepsilon = 1$, penalties off, from random draws of the distribution
   (retried at $\varepsilon = 10$ and $0.1$ if nothing converges).
3. **Continue, in phases.** When a limit is first seen violated, its
   penalty ramps up in decades, from $10^{-6}$ to $10^4$ times a scale
   $\rho_{\text{ref}}$ that makes the penalty of the violation it saw as large as
   the cost, with $\varepsilon$ held. Then $\varepsilon$ falls in decades with the
   penalties held. Steps adapt: they grow by half after a success and halve
   after a failure, and a secant predictor starts each solve. The phases
   matter: on a stiff problem like Goddard's, raising $\rho$ while
   $\varepsilon$ is still falling fails at once.
4. **Read the structure.** After every step, the regularized solution is
   read:
   - each interval where a limit is violated becomes a *boundary arc*, or,
     for order 2, possibly a *touch point*;
   - each stretch where a bang-bang control is strictly between its bounds,
     cut where a limit is violated, becomes a *singular arc* if it is long
     (at least 2% of the horizon), or else is the ramp of an ordinary switch;
   - a stretch that leaves a boundary arc becomes a *joined* singular arc,
     and one that runs onto a limit is just the smoothed approach to it;
   - a reading is *ready* when the ramps not read as singular are short
     (at most 5% of the horizon in all).
5. **Jump.** Each candidate structure is solved exactly with
   `MultiArcShooting` on the exact problem, seeded with the regularized
   costates, the junction times read off, jump multipliers fitted to the
   costates, and a few seeds with jittered times. Each attempt has a budget,
   so a hopeless candidate fails fast.
6. **Accept.** A candidate must converge and pass every check
   `diagnostics` can make: no violation on free arcs, controls within bounds
   on special arcs, $\mu \ge 0$, $\nu \ge 0$, Legendre–Clebsch, and the held
   bounds agreeing with the switching function. Among the candidates
   accepted at one step, the cheapest wins.

Bang-bang switches on free arcs need no reading: they are events of the
integration (Lecture 7). A problem without bang-bang controls or path
constraints is solved by stage 2 alone.

Stages 3 to 5 are the *penalty route*. There is a second route for path
constraints, the *active-set route* of section 6, which replaces them when
the exact problem cannot have singular arcs. Whichever goes first, the other
is its fallback.

In [ ]:
# The stages of Homotopy.solve, and the loop between them.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import itertools

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
fig, ax = plt.subplots(figsize=(12, 3.9))
stages = [
    (0.0, "1. regularize", "smooth bang-bang u,\npenalize each limit"),
    (2.0, "2. sample", "random costates;\nshoot at eps = 1"),
    (4.0, "3. continue", "raise each rho, then\nlower eps, step by step"),
    (6.0, "4. read", "where are the limits hit?\nwhere is u strictly inside?"),
    (8.0, "5. jump", "solve each candidate\nstructure exactly"),
    (10.0, "6. accept", "checks pass:\nkeep the cheapest"),
]
w, hgt = 1.75, 1.2
for x, title, body in stages:
    ax.add_patch(FancyBboxPatch((x, 0), w, hgt, boxstyle="round,pad=0.04", fc="white",
                                ec=ORANGE if title.startswith(("4", "5")) else BLUE, lw=1.6))
    ax.text(x + w / 2, hgt - 0.25, title, ha="center", fontsize=10.5, weight="bold")
    ax.text(x + w / 2, 0.42, body, ha="center", va="center", fontsize=8.5)
for (x0, *_), (x1, *_) in itertools.pairwise(stages):
    ax.annotate("", xy=(x1 - 0.06, hgt / 2), xytext=(x0 + w + 0.06, hgt / 2),
                arrowprops={"arrowstyle": "-|>", "color": "k", "lw": 1.2})
ax.annotate("", xy=(4.0 + w / 2, -0.06), xytext=(8.0 + w / 2, -0.06),
            arrowprops={"arrowstyle": "-|>", "color": ORANGE, "lw": 1.4, "connectionstyle": "arc3,rad=-0.35"})
ax.text(6.9, -1.05, "no candidate accepted: take the next continuation step and read again",
        ha="center", fontsize=9, color=ORANGE)
ax.text(1.0 + w / 2, 1.65, "the problem with smoothing and penalties: no structure to get wrong",
        ha="center", fontsize=9, color=GRAY)
ax.text(9.0 + w / 2, 1.65, "the exact problem: arcs and junctions", ha="center", fontsize=9, color=GRAY)
ax.set(xlim=(-0.2, 12.0), ylim=(-1.3, 1.95))
ax.axis("off")
fig.tight_layout()

*The stages and the loop. Stages 4 and 5 run after every continuation step, not only at the end.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.homotopy import Homotopy, exact, regularize
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.shooting import SingleShooting, parameter_continuation

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

reg, eps, rhos = regularize(examples.goddard(dynamic_pressure=True))
print("regularized parameters:", [str(q) for q in reg.params])
print("running cost (Goddard has none of its own):", reg.running_cost)
print("penalties:", [(c.name, c.penalty) for c in reg.path_constraints])
print("exact problem penalties:", [c.penalty for c in exact(reg).path_constraints])

## 3. What the reader sees

Do stages 2 and 3 by hand for Goddard's rocket (no pressure limit) and look
at a smoothed solution part way down.

In [ ]:
k_reg = build_kernels(derive(regularize(examples.goddard())[0]))
P = list(examples.GODDARD_PARAMS)
s1 = SingleShooting(k_reg, P + [1.0])
first, _ = s1.multistart(
    lambda rng: s1.guess_multipliers(rng.uniform([-6, -0.5, -0.1], [-1, 0, 0.1]),
                                     tf=rng.uniform(0.15, 0.3)), n_tries=50)
path = parameter_continuation(k_reg, P + [1.0], "eps_reg", np.logspace(0, np.log10(0.06), 12),
                              first.v)
fig, ax = plt.subplots(figsize=(7, 3.2))
picks = [path[0], path[len(path) // 3], path[2 * len(path) // 3], path[-1]]
for (e, sol), color in zip(picks, [GRAY, ORANGE, BLUE, "k"]):
    tr = SingleShooting(k_reg, P + [e]).trajectory(sol.v, n_points=600)
    ax.plot(tr["t"] / tr["t"][-1], tr["u"][:, 0], color=color, lw=2, label=f"eps = {e:.2f}")
ax.axhline(3.5, color="k", lw=0.8)
ax.axhline(0.0, color="k", lw=0.8)
ax.set(xlabel="t / t_f", ylabel="smoothed thrust")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# What the reader sees at eps = 0.06, using the package's own reader: the thrust on its
# upper bound, strictly inside for a long stretch, then off. That reads as "upper,
# singular, lower", with rough junction times; the exact solve of that structure (dashed)
# then moves the junctions to where they belong.
hg = Homotopy(examples.goddard())
e_last, sol_last = path[-1]
reading = hg._read(SingleShooting(hg.kr, P + [e_last]), sol_last.v)
candidates, _ = hg._candidates(reading)
items = candidates[0]                                 # the first (most singular) candidate
r_exact = hg.solve(P, ([-6, -0.5, -0.1], [-1, 0, 0.1]), (0.15, 0.3))
te = r_exact.shooting.trajectory(r_exact.solution.v, n_points=1200)

fig, ax = plt.subplots(figsize=(9, 3.6))
t_read, u_read = reading["t"], reading["u"][:, 0]
(_, _, a, b), = [it for it in items if it[0] == "singular"]
for lo, hi, name, color in [(t_read[0], a, "upper", ORANGE), (a, b, "read as singular", BLUE),
                            (b, t_read[-1], "lower", GRAY)]:
    ax.axvspan(lo, hi, color=color, alpha=0.15, lw=0)
    ax.text((lo + hi) / 2, 3.75, name, ha="center", fontsize=9)
ax.plot(t_read, u_read, color=BLUE, lw=2, label=f"smoothed solution, eps = {e_last:.2f}")
ax.plot(te["t"], te["u"][:, 0], color="k", lw=1.2, ls="--", label="exact solve of the reading")
ax.set(xlabel="t", ylabel="thrust u", ylim=(-0.2, 4.1))
ax.legend(frameon=False, fontsize=8, loc="center right")
ax.grid(alpha=0.25)
fig.tight_layout()
print("candidate read:", [(kind, round(a_, 4), round(b_, 4)) for kind, _, a_, b_ in items])
print("exact junctions:", np.round(r_exact.shooting.unpack_arcs(r_exact.solution.v), 4))

*The reading at $\varepsilon = 0.06$, made by the package's own reader. It is rough: the singular stretch it reads spans most of the burn. The exact solve of that structure (dashed) moves the junctions to where they belong. The reading only has to name the arcs and seed their times.*

At $\varepsilon = 1$ the thrust never reaches a bound. Nothing can be read,
and the reading is not ready. By $\varepsilon \approx 0.06$ it starts at full
thrust, ends at zero, and in between spends a long stretch strictly between
the bounds. That reads as "upper, singular, lower", and the exact solve of
that structure converges at once (Lecture 12). The smoothed problem itself
could not be continued much further. Single shooting near a singular arc
grows like $e^{\sqrt{2/\varepsilon}\,t}$, and below $\varepsilon \approx 0.02$
the continuation stalls. The exact structure is better conditioned than any
smoothed problem near it, because the held bounds and the singular control
have no steep ramps to resolve. That is why the jump is attempted at every
step, not at the end.

## 4. The whole solve, logged

`verbose=True` prints each step. The result records everything:

In [ ]:
h = Homotopy(examples.goddard(dynamic_pressure=True), verbose=True)
r = h.solve(examples.GODDARD_Q_PARAMS, ([-6, -0.5, -0.1], [-1, 0, 0.1]), (0.15, 0.3))

In [ ]:
print("exact:     ", r.exact, "|", r.message)
print("structure: ", r.arcs)
print("objective: ", r.objective, " (phi + int L of the exact problem)")
print("draws:     ", r.tries)
print("shooting:  ", r.shooting)
print("junctions: ", r.shooting.unpack_arcs(r.solution.v))
for step in r.steps:
    print(f"   eps decades fallen {step.s:5.2f}  eps {step.eps:8.4f}  rho {np.round(step.rho, 6)}"
          f"  candidates {step.candidates:2d}  accepted {step.accepted}")

Read the log against the stages. The limit is first violated only once
$\varepsilon$ is small enough for the climb to be aggressive. Its penalty
switches on at a millionth of its scale, $\rho = 1.3 \times 10^{-4}$, and
that very first reading is accepted, with the joined structure: even a
barely penalized solution shows where the limit is crossed, and that is all
the exact solve needs. `r.shooting` is the exact `MultiArcShooting`, so every
tool of Lectures 12 and 14 (`trajectory`, `diagnostics`, `cost`) applies to
it. `Homotopy` keeps the compiled kernels, so later solves with other
parameters reuse them; `homotopy.solve(problem, params, costates, ...)` is the
one-call version.

## 5. When it does not succeed

`r.exact` is `False` when no candidate was accepted before the continuation
ended. `r.solution` is then the last regularized solution, on `r.shooting`
(the regularized kernels) at `r.params`, which is often a useful
approximation, and `r.message` says why. The causes, in roughly decreasing
order of frequency:

- **A structure the exact shooting cannot represent**: a limit of order 3
  or more (Lecture 14), two limits active at once, one limit with more than
  one bang-bang control, a singular arc ending directly on a boundary arc.
- **Arcs too short to read**: shorter than the 2% thresholds.
- **A costate distribution that misses**: no draw converges even for the
  regularized problem (`r.tries` equals `n_tries`). Widen or rescale it.

The reading is heuristic and the checks are necessary conditions, so an
accepted structure is an extremal, not a certificate (Lecture 5). Different
draws can lead to different extremals: Lecture 10 found two for the
low-thrust transfer, with propellant 0.167347 and 0.166742. Trying the exact
solve at every step and keeping the cheapest acceptance helps. Below, three
seeds all reach the cheaper one. `seed` and `n_tries` control the draws, and
running several seeds is the honest way to look for others.

In [ ]:
fuel = examples.planar_low_thrust_fuel()
hf = Homotopy(fuel)
p_fuel = [1.0, 0.02, 1.0, 1.0, 1.0, 1.5, 15.0]
for seed in range(3):
    rf = hf.solve(p_fuel, ([-2, -2, -2, -2, 0], [2, 2, 2, 2, 2]), seed=seed)
    print(f"seed {seed}: {rf.message:28s} propellant {rf.objective:.8f}")

## 6. A second route: tightening the limits

The penalty route reads the structure off a smoothed solution, and that
reading can be blind. Take one axis of a machine, jerk-controlled, with
limits on acceleration and speed (`examples.jerk_limited_move`): the
minimum-time move is motion control's *S-curve*, with three boundary arcs.
At $\varepsilon = 1$ the smoothed jerk is so gentle that the acceleration
never reaches its limit, so no reading at that smoothing contains the
acceleration arcs. Lowering $\varepsilon$ while the penalties ramp up is
too stiff to get far, and every jump from a reading misses.

The active-set route works with exact arcs from the start, on the smoothed
problem, where they are well conditioned:

1. **Loosen.** Each limit's offset starts just above the unconstrained
   smoothed solution, so no limit is active.
2. **Tighten.** The offsets fall to zero in adaptive steps, with a secant
   predictor. When a free arc starts to violate a limit, the structure
   changes where it does:
   - a newborn touch point (order 2) or short boundary arc (order 1), with
     zero jump, since a limit that has only just come into play pushes
     with a vanishing multiplier;
   - a touch point violated beside it grows into a boundary arc (keeping
     its jump as the first $\pi$), or splits into two touch points;
   - neighbouring arcs of one limit, violated between them, merge, and
     their jumps add;
   - an arc that shrinks to nothing, or a touch point whose jump turns
     negative, is retired.
3. **Sharpen.** Then $\varepsilon$ falls with the structure following,
   and after every step the *exact* problem is solved with the current
   structure and put to the checks of stage 6.

A violation must be small before the structure changes, so a step that
overshoots is halved. Each change is then a small perturbation of a
solution in hand, the situation where Newton's method is at home.

In [ ]:
from auto_optimal_control.examples import jerk_limited_min_time, jerk_limited_move

hj = Homotopy(jerk_limited_move(), verbose=True)
rj = hj.solve([1.0, 1.0, 2.0, 10.0], ([-3, -3, -3], [3, 3, 3]), (0.5, 30.0))

In [ ]:
print(rj.message, "|", rj.arcs)
print(f"t_f = {rj.solution.tf:.12f}, closed form {jerk_limited_min_time(1.0, 1.0, 2.0, 10.0)[0]}")
trj = rj.shooting.trajectory(rj.solution.v, n_points=1200)
fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharex=True)
for ax, (i, name) in zip(axes, [(0, "position x"), (1, "speed v"), (2, "acceleration a")]):
    ax.plot(trj["t"], trj["z"][:, i], color=BLUE, lw=2)
    ax.set_title(name, fontsize=10, loc="left")
# At a junction itself S = 0 and the sampled jerk is ambiguous: leave those samples out.
at_junction = np.isclose(trj["t"][:, None], trj["junctions"][None, :], atol=1e-9).any(axis=1)
axes[3].plot(np.where(at_junction, np.nan, trj["t"]), trj["u"][:, 0], color=ORANGE, lw=2)
axes[3].set_title("jerk j (the control)", fontsize=10, loc="left")
for ax in axes:
    for c, color in zip(range(3), [GRAY, BLUE, BLUE]):
        on = trj["arc"] == c
        if on.any():
            ax.axvspan(trj["t"][on].min(), trj["t"][on].max(), color=color, alpha=0.12, lw=0)
    ax.set_xlabel("t")
    ax.grid(alpha=0.3)
fig.tight_layout()

The log tells the story: the speed limit comes into play first, as a touch
point that grows into a boundary arc while the offsets fall; the
acceleration arcs are born once $\varepsilon$ has fallen far enough for the
jerk to be aggressive; and at $\varepsilon \approx 0.5$ the exact problem
solves with that structure, to the closed form. The shaded spans are the
boundary arcs: the acceleration limits (blue) and the cruise at the speed
limit (gray), with the jerk zero on all three. The whole solve takes a
couple of seconds.

The route does not read singular arcs, which is why it gives way to the
penalty route when the exact problem may have them, as Goddard's does.

## Summary

- `Homotopy(problem).solve(params, costates, final_time)` needs only a
  costate distribution (a box or `rng -> lam0`) and, for a free final time, a
  `final_time` distribution.
- It regularizes (`regularize`, `exact`), samples, continues in phases
  (penalties ramp with $\varepsilon$ held, then $\varepsilon$ falls), reads the
  structure from each regularized solution, solves the candidates exactly,
  and accepts the cheapest that passes the checks.
- The exact structure is usually solvable long before the smoothing is small,
  so the jump is tried at every step.
- The active-set route tightens offset limits from inactive to exact, with
  exact arcs on the smoothed problem born, grown, split, merged and retired
  as the limits come into play; it finds the S-curve's structure in every
  regime, and gives way to the penalty route when singular arcs are possible.
- `HomotopyResult`: `exact`, `solution`, `shooting`, `arcs`, `objective`,
  `params`, `steps` (`Step`: `s`, `eps`, `rho`, `residual`, `candidates`,
  `accepted`), `tries`, `message`.

## Exercises

1. Run the train of Lecture 14 (`speed_limited_min_time`) with
   `verbose=True` and explain each line of the log.
2. Give Goddard a costate box that does *not* contain the solution's costates
   (for instance positive $\lambda_h$). What happens, and what does
   `r.message` say?
3. Write a two-limit problem, a double integrator with both $|v| \le V$ and
   $x \le \ell$, where both limits become active at the same time. Run
   `Homotopy` on it and read the result. Which limitation of section 5 is
   it?

## Further reading

- O. von Stryk and R. Bulirsch, "Direct and indirect methods for trajectory
  optimization", *Annals of Operations Research* 37, 1992.
- H. J. Oberle and W. Grimm, "BNDSCO: a program for the numerical solution of
  optimal control problems", DLR report IB 515-89/22 (1989).
- The module documentation of `auto_optimal_control.homotopy`, which lists
  every threshold.